For faster downloads and for better GPU, use colab extension on VS Code to connect to a remote Colab Instance

# Checking the Non domain-adapted model

ModernBERT-base is used for its benchmarks and size.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments, default_data_collator, DataCollatorForLanguageModeling
from datasets import Dataset
import collections
import numpy as np
import math

SEED = 67
CHUNK_SIZE = 128
WWM_PROB = 0.2
BATCH_SIZE = 64 # GPU bound

In [15]:

model_use = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_use)
model = AutoModelForMaskedLM.from_pretrained(model_use)

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

Attempt on an arbitrary text on a news body

In [16]:
test = " Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week [MASK] as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte\u2019s desk"

In [17]:
inputs = tokenizer(test, return_tensors='pt')
outputs = model(**inputs)
logits = outputs.logits

In [18]:
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = logits[0, mask_token_index, :]
top_10_tokens = torch.topk(mask_token_logits, 10, dim = 1).indices[0].tolist()
for token in top_10_tokens:
    print(f"{tokenizer.decode([token])}, {test.replace(tokenizer.mask_token, tokenizer.decode(token).strip())}")

 stint,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week stint as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 tenure,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week tenure as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 term,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week term as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 assignment,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week assignment as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodri

## Preprocessing the Dataset

In [19]:
import jsonl
import pandas as pd



titles = []
articles = []

sources = ["abscbn", "gma", "rappler", "upparser"]
for source in sources:
    with open(f'../scraping/{source}_articles.jsonl', encoding='utf-8', errors="replace") as source_f:
        source_json = jsonl.load(source_f)
        for article in source_json:
            t = article['title']
            b = article['body']
            # print(b)
            titles.append(t)
            articles.append(b)

articles_df = pd.DataFrame({
    "title": titles,
    "body": articles
})

articles_df

,title,body
0,Iran minister heads to Russia as talks remain ...,Iran's foreign minister headed to Russia on Su...
1,Dizon orders immediate declogging of SLEX drai...,Dizon said the DPWH will work together with SL...
2,Marcos to tackle China's gray zone tactics wit...,Marcos said he wants to find out more about 't...
3,11 of 19 Toboso fatalities tested positive for...,"Police Colonel Reynaldo Calaoa, chief of the P..."
4,"Leviste unhappy in Congress, open to other pos...",Speaking at the Kapihan ng Samahang Plaridel i...
...,...,...
34528,The UP Parser Receives Best Student Publicatio...,The UP Parser was awarded twice during the Eng...
34529,"Bootcamp 5.0: Freshies, Assemble!",Bootcamp 5.0 was held at the UP Department of ...
34530,CURSOR Week,Episode 2 of 10 (1516A): In this episode of CS...
34531,UP CURSOR Launches its Newest Logo,The UP Association of Computer Science Majors ...


In [20]:
def tokenize_func(data):
    tokenized = tokenizer(data["body"])
    if tokenizer.is_fast:
        tokenized["word_ids"] = [tokenized.word_ids(i) for i in range(len(tokenized["input_ids"]))]
    return tokenized


articles_ds = Dataset.from_pandas(articles_df)
articles_ds = articles_ds.train_test_split(seed=SEED)

tokenized_articles = articles_ds.map(
    tokenize_func, batched=True, remove_columns=['title', 'body']
)
tokenized_articles


Map:   0%|          | 0/25899 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (9596 > 8192). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/8634 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids'],
        num_rows: 25899
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids'],
        num_rows: 8634
    })
})

Concatenate all articles and split each article into texts of length chunk size.

In [23]:
def concatenate_chunk(data):
    concatenated = {key: sum(data[key], []) for key in data}
    total_length = len(concatenated[list(data)[0]])
    # remove last when smaller than chunk size
    total_length = (total_length // CHUNK_SIZE) * CHUNK_SIZE
    # split
    splitted = {
        key: [row[i:i + CHUNK_SIZE] for i in range(0, total_length, CHUNK_SIZE)]
        for key, row in concatenated.items()
    }

    splitted["labels"] = splitted["input_ids"].copy()

    return splitted

In [24]:
articles_chunked = tokenized_articles.map(concatenate_chunk, batched=True)
articles_chunked

Map:   0%|          | 0/25899 [00:00<?, ? examples/s]

Map:   0%|          | 0/8634 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 136604
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 46421
    })
})

In [25]:
articles_chunked.save_to_disk('articles_chunked_dataset')

Saving the dataset (0/1 shards):   0%|          | 0/136604 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/46421 [00:00<?, ? examples/s]

In [10]:
tokenizer.decode(articles_chunked["train"][0]["input_ids"])

"[CLS]Ayon sa driver ng 12-wheeler truck, patungo siya sa Pasig mula Pampanga at may kargang buhangin nang makarinig siya ng kalabog. 'Naramdaman ko na lang po may kumalabog sa ilalim ng truck ko. Dito po siya sa pinakakanan ko, sa mismong island,' paliwanag ng driver. Agad niyang inihinto ang trak at doon na nakita ang nagulungang rider. Dead-on-the-spot ang biktima"

## Fine-tuning

### Training

In [12]:
logging_steps = len(articles_chunked["train"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

training_args = TrainingArguments(
    output_dir=f"{model_use.split('/')[-1]}-finetuned-all",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_eval_batch_size=BATCH_SIZE,
    per_device_train_batch_size=BATCH_SIZE,
    fp16=True,
    logging_steps=logging_steps
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=articles_chunked["train"],
    eval_dataset=articles_chunked["test"],
    data_collator=data_collator,
)

In [14]:
eval_results = trainer.evaluate()
print(f"Perplexity before fine-tuning: {math.exp(eval_results['eval_loss'])}")

Perplexity before fine-tuning: 8.539410216555021


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,No log,1.457739,0.002600
2,No log,1.400394,0.002600
3,No log,1.387698,0.002600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6405, training_loss=1.4415251756440282, metrics={'train_runtime': 6870.9014, 'train_samples_per_second': 59.645, 'train_steps_per_second': 0.932, 'total_flos': 3.492703213800653e+16, 'train_loss': 1.4415251756440282, 'epoch': 3.0})

In [16]:
eval_results = trainer.evaluate()
print(f"Perplexity after fine-tuning: {math.exp(eval_results['eval_loss'])}")

Perplexity after fine-tuning: 4.022210057930634


In [17]:
trainer.save_model() 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [1]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch


tokenizer_test = AutoTokenizer.from_pretrained('./ModernBERT-base-finetuned-all')
model_test = AutoModelForMaskedLM.from_pretrained('./ModernBERT-base-finetuned-all', dtype=torch.float16)


test = "CS Week 2025 [MASK] with a blast as students conquer the halls with glee"
test = " Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week [MASK] as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte\u2019s desk"
test_input = tokenizer_test(test, return_tensors="pt")

test_output = model_test(**test_input)
test_logits = test_output.logits


mask_token_index = torch.where(test_input["input_ids"] == tokenizer_test.mask_token_id)[1]
mask_token_logits = test_logits[0, mask_token_index, :]
top_10_tokens = torch.topk(mask_token_logits, 10, dim = 1).indices[0].tolist()
for token in top_10_tokens:
    print(f"{tokenizer_test.decode([token])}, {test.replace(tokenizer_test.mask_token, tokenizer_test.decode(token).strip())}")

top_10 = torch.argsort(mask_token_logits, dim = 1, descending=True)
print(top_10)

for t in top_10.tolist():
    print(t)

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

 stint,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week stint as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 term,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week term as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 tenure,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week tenure as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 visit,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week visit as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte

# Metrics

Functions for evaluation and metrics

In [1]:
def preprocess_logits_for_metrics(logits, labels):
    preds = logits.argmax(-1)
    topk = torch.topk(logits, k=5, dim=-1).indices
    return preds, topk


def compute_metrics(eval_pred):
    (preds, topk), labels = eval_pred
    mask = labels != -100
    acc = (preds[mask] == labels[mask]).mean()
    topk_acc = (topk[mask] == labels[mask][..., None]).any(-1).mean()

    return {
        "masked_acc": float(acc),
        "masked_top5": float(topk_acc)
    }
    

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments, default_data_collator, DataCollatorForLanguageModeling
from datasets import load_from_disk
import numpy as np
import math

SEED = 67
BATCH_SIZE_EVAL = 64 # GPU bound

In [3]:
model_use = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_use)
model = AutoModelForMaskedLM.from_pretrained(model_use)


tokenizer_finetuned = AutoTokenizer.from_pretrained('./ModernBERT-base-finetuned-all')
model_finetuned = AutoModelForMaskedLM.from_pretrained('./ModernBERT-base-finetuned-all')

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

In [4]:
ds = load_from_disk("./articles_chunked_dataset")

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15, seed=SEED)

In [5]:
args = TrainingArguments(output_dir="./eval_test",eval_accumulation_steps=4 ,per_device_eval_batch_size=BATCH_SIZE_EVAL, fp16=True, seed=SEED)

trainer_finetuned = Trainer(
    model=model_finetuned,
    args=args,
    eval_dataset=ds['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

trainer_base = Trainer(
    model=model,
    args=args,
    eval_dataset=ds['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

In [6]:
metrics_base = trainer_base.evaluate()
metrics_finetuned = trainer_finetuned.evaluate()


Training Loss,Validation Loss,Step,Masked Acc,Masked Top5
No log,2.143805,0,0.643859,0.803399


Training Loss,Validation Loss,Step,Masked Acc,Masked Top5
No log,1.387652,0,0.704473,0.855017


In [8]:
print(metrics_base)
print(metrics_finetuned)

{'eval_loss': 2.1438050270080566, 'eval_masked_acc': 0.6438591832647357, 'eval_masked_top5': 0.8033986754549608}
{'eval_loss': 1.3876515626907349, 'eval_masked_acc': 0.704473404547806, 'eval_masked_top5': 0.8550169395282967}


In [17]:
import pandas as pd
import plotly.express as px

def perplexity(metric):
    return math.exp(metric['eval_loss'])

metrics_base['perplexity'] = perplexity(metrics_base)
metrics_finetuned['perplexity'] = perplexity(metrics_finetuned)

metrics_df = pd.DataFrame(
    {
        "base": metrics_base,
        "finetuned": metrics_finetuned
    }
)

In [25]:
metrics_df

,base,finetuned
eval_loss,2.143805,1.387652
eval_masked_acc,0.643859,0.704473
eval_masked_top5,0.803399,0.855017
perplexity,8.531840,4.005432


In [30]:
fig = px.bar(metrics_df.loc[['eval_masked_acc', 'eval_masked_top5']], barmode="group")
fig.update_xaxes(title="Metric per Model")
fig.update_legends(title="Model")
fig.update_traces(
    texttemplate="%{y:.4f}",
)
fig.show()

In [31]:
fig = px.bar(metrics_df.loc[['eval_loss']], barmode="group")
fig.update_xaxes(title="Metric per Model")
fig.update_legends(title="Model")
fig.update_traces(
    texttemplate="%{y:.4f}",
)
fig.show()

In [32]:
fig = px.bar(metrics_df.loc[['perplexity']], barmode="group")
fig.update_xaxes(title="Metric per Model")
fig.update_legends(title="Model")
fig.update_traces(
    texttemplate="%{y:.4f}",
)
fig.show()

In [54]:
# Epoch	Training Loss	Validation Loss	Model Preparation Time
# 1	No log	1.457739	0.002600
# 2	No log	1.400394	0.002600
# 3	No log	1.387698	0.002600
loss = [2.143805 ,1.457739, 1.400394, 1.387698]

fig = px.line(loss, range_x=(0,3), title="Loss per Epoch")
fig.update_xaxes(title="Epoch")
fig.update_yaxes(title="Loss")
fig.update_legends(title="Loss")
fig.show()